# DLT Pipeline

#### Streaming Table

 We will create a streaming table on source silver.products table(data)

In [0]:
import dlt
from pyspark.sql.functions import *

In [0]:
# Expectations
my_rules = {
    "rule1" : "product_id IS NOT NULL",
    "rule2" : "product_name IS NOT NULL"
}

In [0]:
@dlt.table()  # If we give the name here, it will be the table name. If we didn't function name will be taken as the table name

@dlt.expect_all_or_drop(my_rules)
def DimProducts_stage():
    df = spark.readStream.table("databricks_wspace.silver.products_silver")   # .table - bcs we have a table
    return df  # the dataframe will be returned and will be created a streaming table on top of this dataframe

Now our streaming table is ready. Let's create our streaming view on top of it

#### Streaming View

In [0]:
@dlt.view

def DimProducts_view():
    df = spark.readStream.table("Live.DimProducts_stage")  # we use the 'Live' keyword in dlt to refer to the tables even if they are not created(ran) yet
    return df

#### DimProducts

In [0]:
dlt.create_streaming_table("DimProducts")  # this will create a empty streaming table and we call it as DimProducts

In [0]:
dlt.apply_changes(
    target = "DimProducts",
    source = "Live.DimProducts_view",
    keys = ["product_id"],
    sequence_by = "product_id",
    stored_as_scd_type = 2
)

# target = the empty streaming table we just created
# source = view we created
# keys = primary key
# sequence_by = date column. but sinc ewe dont have we'll give the primary key

- SCD Type-2 is created. Next to creating pipeline